In [1]:
import sys
from pathlib import Path

# project/
project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

In [9]:
from collections import defaultdict
from pathlib import Path
import pickle

from rank_bm25 import BM25Okapi
from week5_ragfoundations.day4.retrival import retrieve
from sentence_transformers import CrossEncoder

In [3]:
#bm25 retrival
BASE_DIR = Path.cwd()

METADATA_PATH = (
    BASE_DIR.parent.parent
    / "week5_ragfoundations"
    / "day4"
    / "metadata.pkl"
)
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

# print(metadata)

documents = [
    item["text"]
    for item in metadata
]


# print(documents)


tokenized_docs = [
    doc.lower().split()
    for doc in documents
]


bm25 = BM25Okapi(tokenized_docs)

query = input("Enter your query: ")

tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)

top_indices = sorted(
    range(len(scores)),
    key=lambda i: scores[i],
    reverse=True
)[:20]

res = []

for rank, idx in enumerate(top_indices, start=1):
    res.append({
        "chunk_id": metadata[idx]["chunk_id"],
        "text": metadata[idx]["text"],
        "source": metadata[idx]["source"],
        "rank": rank
    })
res

[{'chunk_id': 0,
  'text': 'Report of the Central Board of Directors on the working of the Reserve Bank of India\nfor the year ended March 31, 2026 submitted to the Central Government in terms of \nSection 53(2) of the Reserve Bank of India Act, 1934\nRESERVE BANK OF INDIA ANNUAL REPORT\n2025-26\nGOVERNOR\nSanjay Malhotra\nDEPUTY GOVERNORS\nSwaminathan J.\nPoonam Gupta\nShirish Chandra Murmu\nRohit Jain\nDIRECTORS NOMINATED UNDER \nSECTION 8 (1) (b) OF THE RBI ACT, 1934\nRevathy Iyer\nSachin Chaturvedi\nDIRECTORS NOMINATED UNDER',
  'source': 'rbireport.pdf',
  'rank': 1},
 {'chunk_id': 1,
  'text': 'Sachin Chaturvedi\nDIRECTORS NOMINATED UNDER \nSECTION 8 (1) (c) OF THE RBI ACT, 1934\nSatish Kashinath Marathe \nSwaminathan Gurumurthy\nAnand Gopal Mahindra\nVenu Srinivasan\nPankaj Ramanbhai Patel\nRavindra H. Dholakia\nDIRECTORS NOMINATED UNDER \nSECTION 8 (1) (d) OF THE RBI ACT, 1934\nNagaraju Maddirala\nAnuradha Thakur\nMEMBERS OF LOCAL BOARDS\n \nWESTERN AREA\nEASTERN AREA\nSachin C

In [4]:
#RRF reciprocal rank fusion

def reciprocal_rank_fusion(result_lists, k=60):
    scores = defaultdict(float)
    chunks = {}

    for results in result_lists:

        for rank, chunk in enumerate(results):

            chunk_id = chunk["chunk_id"]

            scores[chunk_id] += 1 / (k + rank + 1)

            chunks[chunk_id] = chunk

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    fused_results = []

    for chunk_id, _ in ranked:
        fused_results.append(chunks[chunk_id])

    return fused_results

In [5]:
def hybrid_retrieve(query, top_k=5):

    dense_results = retrieve(query, top_k=20)

    bm25_results = res

    final_results = reciprocal_rank_fusion([
        dense_results,
        bm25_results
    ])

    return final_results[:top_k]

In [6]:
dense_results = retrieve("spacecraft", top_k=20)
bm25_results = res

final_results = reciprocal_rank_fusion(
    [dense_results, bm25_results]
)

dense_results
final_results

[{'chunk_id': 3,
  'text': '.......................................................................\n  .......................................................................\n  .......................................................................\n  .......................................................................\n  .......................................................................\n  .......................................................................',
  'source': 'rbireport.pdf',
  'rank': 4},
 {'chunk_id': 4,
  'text': '.......................................................................\n  .......................................................................\n  .......................................................................\n  ....................................................................... \n  .......................................................................\nVivek Deep\nRadha Shyam Ratho \nAjay Kumar\nNeeraj Nigam\nP. Vasudevan\nR. Lak

In [10]:
#iske pehle jo bhi tha sab bi-enocder tha

#cross encoder
model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

c:\Users\ASUS\PycharmProjects\Machine-learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11670.91it/s]


In [14]:
def crossencoding(query):
    pairs = [(query, i['text']) for i in final_results]
    # print(pairs)
    scores = model.predict(pairs)
    print(scores)
    ranked = sorted(
        zip(final_results, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return ranked[:5]

crossencoding("rbi report")

[ -6.397071   -7.598524  -11.244698   -9.690436    1.3114088 -10.145115
  -1.9841123  -9.924566   -6.928754  -10.351606   -7.3465714  -7.8412285
  -8.918213   -8.381132  -10.685979   -7.710661  -10.557047   -7.8017173
  -7.1982183  -8.548896   -8.367023  -10.731156   -9.63516    -9.008183
  -9.970865  -10.195712   -9.11112    -9.402573   -7.663155   -2.1756938
  -7.873722   -7.0456696  -7.5255723  -9.134811   -7.5191026  -9.317539
  -7.2533293]


[({'chunk_id': 0,
   'text': 'Report of the Central Board of Directors on the working of the Reserve Bank of India\nfor the year ended March 31, 2026 submitted to the Central Government in terms of \nSection 53(2) of the Reserve Bank of India Act, 1934\nRESERVE BANK OF INDIA ANNUAL REPORT\n2025-26\nGOVERNOR\nSanjay Malhotra\nDEPUTY GOVERNORS\nSwaminathan J.\nPoonam Gupta\nShirish Chandra Murmu\nRohit Jain\nDIRECTORS NOMINATED UNDER \nSECTION 8 (1) (b) OF THE RBI ACT, 1934\nRevathy Iyer\nSachin Chaturvedi\nDIRECTORS NOMINATED UNDER',
   'source': 'rbireport.pdf',
   'rank': 1},
  np.float32(1.3114088)),
 ({'chunk_id': 1,
   'text': 'Sachin Chaturvedi\nDIRECTORS NOMINATED UNDER \nSECTION 8 (1) (c) OF THE RBI ACT, 1934\nSatish Kashinath Marathe \nSwaminathan Gurumurthy\nAnand Gopal Mahindra\nVenu Srinivasan\nPankaj Ramanbhai Patel\nRavindra H. Dholakia\nDIRECTORS NOMINATED UNDER \nSECTION 8 (1) (d) OF THE RBI ACT, 1934\nNagaraju Maddirala\nAnuradha Thakur\nMEMBERS OF LOCAL BOARDS\n \nWEST

In [16]:
from dotenv import load_dotenv
import os

from groq import Groq

load_dotenv()
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [17]:
def build_prompt(question, retrieved_chunks):

    context = ""

    for chunk in retrieved_chunks:

        context += (
            f"Source: {chunk['source']}\n"
            f"{chunk['text']}\n\n"
        )

    prompt = f"""
            You are a helpful AI assistant.

            Answer Only using the context below.

            If the answer is not present in the context,
            reply exactly:

            "I don't know."

            Context
            --------------------
            {context}
            --------------------

            Question:
            {question}

            Answer:
            """

    return prompt

In [18]:
def generate_answer(prompt):

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content


In [22]:
def rag_query(question):

    ranked_chunks = crossencoding(question)
    chunks = [chunk for chunk, score in ranked_chunks]

    prompt = build_prompt(question, chunks)

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources": list(set(chunk["source"] for chunk in chunks)),
        "metadata": list(set((chunk["chunk_id"]) for chunk in chunks))
    }


In [23]:
while True:

        question = input("\nAsk a question (type 'exit' to quit): ")

        if question.lower() == "exit":
            break

        result = rag_query(question)

        print("\nAnswer")
        print("=" * 60)
        print(result["answer"])

        print("\nSources")
        print("=" * 60)

        for source in result["sources"]:
            print(f"- {source}")
        
        print(result)

[ -6.397071   -7.598524  -11.244698   -9.690436    1.3114088 -10.145115
  -1.9841123  -9.924566   -6.928754  -10.351606   -7.3465714  -7.8412285
  -8.918213   -8.381132  -10.685979   -7.710661  -10.557047   -7.8017173
  -7.1982183  -8.548896   -8.367023  -10.731156   -9.63516    -9.008183
  -9.970865  -10.195712   -9.11112    -9.402573   -7.663155   -2.1756938
  -7.873722   -7.0456696  -7.5255723  -9.134811   -7.5191026  -9.317539
  -7.2533293]

Answer
The RBI report is for the year ended March 31, 2026, and is titled "Reserve Bank of India Annual Report 2025-26".

Sources
- rbireport.pdf
{'question': 'rbi report', 'answer': 'The RBI report is for the year ended March 31, 2026, and is titled "Reserve Bank of India Annual Report 2025-26".', 'sources': ['rbireport.pdf'], 'metadata': [0, 1, 2, 3, 778]}
[ -7.95848   -8.49556  -11.408681 -11.229326  -5.372017 -11.178961
  -8.29929  -11.137808  -8.563486 -10.956132 -10.316761  -9.295889
 -10.810438 -10.068752 -11.29451   -9.784425 -11.011425